# SETTLED FAILURE BANK -- clean-data CRL, alpha=0.1 (Colab GPU)

Identical to the Part-1 alpha=0.1 failure-negative run
(`failneg_clean_p30_h800_resetfix_a01_s0_300k`) in every respect -- clean
dataset, recipe, seed, steps, eval protocol -- EXCEPT the failure bank:

    legacy bank (16 healthy-looking frozen death poses)
        -> settled bank (16 N=80 physically-settled fatal observations)

The settled states come from the SAME 16 authoritative pilot death episodes,
re-collected under the death-settle physics patch (actor control zeroed at
the fatal contact, 80 extra MuJoCo substeps inside the fatal transition; see
scripts/rebuild_failure_bank_settled.py + bank_manifest.json: pre-contact
prefixes reproduce the pilot bitwise). The 40 fresh diagnostic deaths are
HELD OUT and never touch this run.

Loss (unchanged): L = L_pos + (1-alpha) L_ordinary-neg + alpha L_failure-neg,
alpha = 0.1, exact uniform expectation over the 16-state bank.


## 1. Configuration -- single source of truth

In [ ]:
# 1. Configuration. Everything tunable lives here.
import os

SEED        = 0
MAX_UPDATES = 300_000
RESUME      = False           # set True after a Colab disconnect (restores latest.pkl)
REQUIRE_GPU = True
P_ACTIVE    = 0.3          # mask density (must match the dataset's collection density)
RESET_FIX   = True        # canonical episode-independent reset (resetfix_v1)
HORIZON     = 800         # H=800 protocol (per Part 1 experiment decision)

# --- failure-aware negative sampling (THE experimental variable) ---------------
ALPHA       = 0.1          # mixture weight on the critic NEGATIVE distribution
                             # 0 => clean baseline (experiment A); >0 => experiment B

# --- repo (pinned to the failure-split commit) ---------------------------------
REPO     = 'contrastive_rl'
REPO_URL = 'https://github.com/tingrui-huang/contrastive_rl.git'
BRANCH   = 'feature/continuous-action-agreement'
COMMIT   = ''

# --- dataset: D_clean = D_original minus the 16 rockfall-death episodes --------
# (ships WITH the repo; the failure bank = 16 terminal buried poses of the
#  removed episodes, used ONLY as biased negatives, never as anchors/positives)
ENV_NAME             = 'offline_ant_umaze_rockfall'
DATASET_REPO_RELPATH = 'artifacts/rockfall_v2_p30_h800_resetfix/failure_split/antmaze_rockfall_v2_p30_h800_resetfix_pilot_clean.npz'
DATASET_SHA256       = '6bec8a52e771569c4edc14ff0c7319df4322fe6d74e6e69a6a7074fc76be1852'
BANK_REPO_RELPATH    = 'artifacts/settled_failure_bank_alpha01/failure_bank_settled.npz'
BANK_SHA256          = '8c50202403317e8adc67670be716fffe5c3b3cb891017bfbe6eb070acd61d0ce'
LEGACY_BANK_SHA256   = '8d35b76ada59199e6ba22250a02bbdda931ff885beb08c2d8d146ecd14b41481'
NEW_MANIFEST_RELPATH = 'artifacts/settled_failure_bank_alpha01/bank_manifest.json'
MANIFEST_RELPATH     = 'artifacts/rockfall_v2_p30_h800_resetfix/failure_split/failure_split_manifest.json'
SOURCE_DATASET_SHA256 = '08bdc44bb7942793ea18c2628c36d8ea8bfe875169afadfa8934b1b948ef55e4'  # the authoritative D_original this was split from

# --- run dirs --------------------------------------------------------------------
RUN_ID            = f'failneg_settledbank_p30_h800_resetfix_a01_s{SEED}_{MAX_UPDATES//1000}k'
LOCAL_RUN_DIR     = f'/content/runs/{RUN_ID}'
RUN_DRIVE_DIR     = f'/content/drive/MyDrive/contrastive_rl_runs/{RUN_ID}'
LOCAL_DATASET_PATH = f'/content/{REPO}/{DATASET_REPO_RELPATH}'
LOCAL_BANK_PATH    = f'/content/{REPO}/{BANK_REPO_RELPATH}'
print(RUN_ID, '| alpha =', ALPHA)


## 2. Mount Google Drive; create dirs

In [ ]:
# 2. Mount Drive; make local scratch + Drive run trees.
# (The rockfall dataset ships WITH the repo -- Drive is only for checkpoint
#  mirroring, so there is no dataset dir to create here.)
from google.colab import drive
drive.mount('/content/drive')
for base in (LOCAL_RUN_DIR, RUN_DRIVE_DIR):
    os.makedirs(base, exist_ok=True)
print('local scratch:', LOCAL_RUN_DIR)
print('Drive run dir:', RUN_DRIVE_DIR)
!df -h /content | tail -1


## 3. Clone/checkout the repo (refuses to overwrite uncommitted work)

In [ ]:
# 3. Clone if absent; otherwise fetch + checkout at COMMIT (or origin/BRANCH).
import subprocess, sys
os.chdir('/content')
if not os.path.exists(REPO):
    !git clone $REPO_URL $REPO
os.chdir(REPO)
dirty = subprocess.run(['git','status','--porcelain'], capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError('repo has uncommitted changes -- refusing to checkout over them:\n' + dirty)
!git fetch -q origin
ref = COMMIT if COMMIT else f'origin/{BRANCH}'
!git checkout -q $ref
!git log -1 --oneline
for req in ('crl/offline_audit.py','crl/rockfall_ant.py','crl/train.py',
            'scripts/naive_rockfall_v2_crl.py','scripts/verify_offline_d4rl.py',
            'scripts/naive_rockfall_v2_crl.py',
            'scripts/diagnose_naive_rockfall.py'):
    if not os.path.exists(req):
        raise RuntimeError(f'{req} missing -- push main from the workstation and rerun.')
print('rockfall pipeline files present -- checkout OK')

## 4. Dependencies (preserve Colab's preinstalled GPU JAX -- do not alter)

In [ ]:
# 4. Install deps WITHOUT disturbing Colab's GPU JAX (pin jax/jaxlib/numpy).
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['MUJOCO_GL'] = 'egl'
import jax, jaxlib, numpy
hold = [f'jax=={jax.__version__}', f'jaxlib=={jaxlib.__version__}', f'numpy=={numpy.__version__}']
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
print('Colab JAX', jax.__version__, '| devices:', jax.devices())
pip('--no-deps', 'dm-haiku', 'optax', 'chex')
pip('jmp', 'tabulate', 'toolz', 'etils', 'tensorboardX', 'mujoco', 'imageio', 'imageio-ffmpeg', *hold)
print('post-install JAX', jax.__version__, '| devices:', jax.devices())

## 5. GPU / environment verification

In [ ]:
# 5. Require an accelerator; record env meta to Drive.
import hashlib, json, platform, mujoco
os.chdir('/content/'+REPO)
commit = subprocess.run(['git','rev-parse','HEAD'], capture_output=True, text=True).stdout.strip()
meta = {'run_id': RUN_ID, 'python': platform.python_version(), 'jax': jax.__version__,
        'backend': jax.default_backend(), 'devices':[str(d) for d in jax.devices()],
        'mujoco': mujoco.__version__, 'git_commit': commit, 'env': ENV_NAME}
if REQUIRE_GPU and jax.default_backend() == 'cpu':
    raise RuntimeError('no accelerator -- Runtime > Change runtime type > GPU')
!nvidia-smi -L || true
json.dump(meta, open(f'{RUN_DRIVE_DIR}/meta_env.json','w'), indent=2)
print(json.dumps(meta, indent=1))

In [ ]:
# 5b. Reset-fix provenance verification (Colab-safe -- no re-run).
# The full reset correctness tests (A-E) need the scripted controllers +
# a naive checkpoint that are not on Colab; they were RUN + COMMITTED on the
# workstation at cap-700 AND cap-800. Here we verify the deployed commit
# carries the exact validated fix.
os.chdir('/content/'+REPO)
import subprocess, json as _j
commit = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
print('checked-out commit', commit)
src = open('crl/rockfall_ant.py').read() + open('crl/d4rl_ant.py').read()
assert 'mj_resetData' in src, 'reset fix (mj_resetData) NOT present in checkout'
rt = _j.load(open('artifacts/reset_fix/reset_tests_h800.json'))
assert rt['reset_fix_version']=='resetfix_v1', 'reset-fix version mismatch'
assert rt['ALL_CORRECTED_PASS'], 'committed reset tests did NOT all pass'
assert rt['cap']==800, 'committed reset tests not at cap-800'
print('reset-fix VERIFIED: version', rt['reset_fix_version'],
      '| cap', rt['cap'], '| A-E all pass (workstation-run, committed)')


## 6. Dataset: verify the repo-shipped CLEAN npz (sha256)

D_clean (284 episodes) + failure bank are committed at the pinned commit -- nothing to upload.


In [ ]:
# 6. Verify the repo-shipped rockfall npz; fail fast on any mismatch.
def sha256(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
if not os.path.exists(LOCAL_DATASET_PATH):
    raise FileNotFoundError('dataset missing from the clone: ' + LOCAL_DATASET_PATH
                            + ' (is COMMIT pinned to 4dd0f8f or later?)')
got = sha256(LOCAL_DATASET_PATH)
if got != DATASET_SHA256:
    raise SystemExit('dataset sha mismatch: got ' + got + ' exp ' + DATASET_SHA256)
print('dataset OK:', LOCAL_DATASET_PATH)
print('sha256:', got[:16], '...')


In [ ]:
# 6b. Verify the SETTLED failure bank + rebuild manifest + split provenance.
import json as _j
got = sha256(f'/content/{REPO}/{BANK_REPO_RELPATH}')
if got != BANK_SHA256:
    raise SystemExit('settled failure-bank sha mismatch: got ' + got)
bman = _j.load(open(f'/content/{REPO}/{NEW_MANIFEST_RELPATH}'))
assert all(bman['checks'].values()), ('bank rebuild validation not clean: '
    + str({k: v for k, v in bman['checks'].items() if not v}))
assert bman['bank']['sha256'] == BANK_SHA256
assert bman['bank']['n_states'] == 16 and bman['bank']['state_dim'] == 29
assert bman['bank']['death_settle_substeps'] == 80
assert bman['old_bank']['sha256'] == LEGACY_BANK_SHA256, 'legacy bank drifted'
assert bman['clean_npz']['sha256'] == DATASET_SHA256, 'clean npz sha mismatch'
assert bman['heldout_fresh_deaths']['n'] == 40, 'held-out list missing'
man = _j.load(open(f'/content/{REPO}/{MANIFEST_RELPATH}'))
assert man['clean']['sha256'] == DATASET_SHA256, 'split clean sha mismatch'
assert man['source_sha256'] == SOURCE_DATASET_SHA256, 'split source mismatch'
assert man['clean']['n_episodes'] == 284 and man['rockfail']['n_episodes'] == 16
assert bman['per_state_provenance'][0].keys() >= {'episode_id',
    'prefix_bitwise_ok'}
assert all(r['prefix_bitwise_ok'] and r['settled_matches_sweep_trace']
           and r['differs_from_legacy'] for r in bman['per_state_provenance'])
print('settled bank verified: 16 N=80 states of the original pilot deaths; '
      'clean npz + legacy bank untouched; 40 fresh deaths held out')


## 7. Pre-training offline audit -- must PASS before any training

In [ ]:
# 7. Static offline gates (G1-G8) on the REAL rockfall dataset.
os.chdir('/content/'+REPO); sys.path.insert(0, '/content/'+REPO)
from crl import offline_audit
from crl.config import Config
from crl import envs as envs_mod
_c = Config(env_name=ENV_NAME, offline_dataset=LOCAL_DATASET_PATH)
envs_mod.make_env(ENV_NAME, _c, seed=0)   # fills obs/goal/action dims
_c.max_episode_steps = HORIZON   # H=800: audit the 801 contract
passed, gates, rep = offline_audit.run_static_audit(LOCAL_DATASET_PATH, _c)
print('offline_audit gates:', {g:('PASS' if ok else 'FAIL') for g,ok in gates.items()})
fp = rep['fingerprint']
print(f"  sha256={fp['sha256'][:16]}  eps={fp['n_episodes']}  trans={fp['n_transitions']}  obs={fp['obs_shape']}")
assert passed, 'offline_audit FAILED -- see gates'

## 8. (resume) restore latest.pkl from Drive

In [ ]:
# 8. If RESUME, pull the last checkpoint from Drive into local scratch.
import glob, shutil, os
if RESUME:
    for f in glob.glob(f'{RUN_DRIVE_DIR}/*.pkl') + [f'{RUN_DRIVE_DIR}/metrics.json',
                                                    f'{RUN_DRIVE_DIR}/offline_dataset.sha256']:
        if os.path.exists(f): shutil.copy2(f, f'{LOCAL_RUN_DIR}/{os.path.basename(f)}')
    print('restored:', sorted(os.path.basename(p) for p in glob.glob(f'{LOCAL_RUN_DIR}/*.pkl')))
else:
    print('fresh run (RESUME=False)')

## 9. TensorBoard

In [ ]:
tb = f'{LOCAL_RUN_DIR}/tb'
%load_ext tensorboard
%tensorboard --logdir $tb

## 10. Launch training (live stream)

Trains to Drive-mirrored local scratch. Eval every 10k in the rockfall env.

In [ ]:
# 10. Launch naive offline CRL on the rockfall dataset (LIVE streaming).
os.chdir('/content/'+REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
cmd = [sys.executable, '-u', 'scripts/naive_rockfall_v2_crl.py',
       '--npz', LOCAL_DATASET_PATH, '--steps', str(MAX_UPDATES),
       '--seed', str(SEED), '--ckpt-dir', LOCAL_RUN_DIR,
              '--p-active', str(P_ACTIVE)] + (['--reset-fix'] if RESET_FIX else []) + ['--horizon', str(HORIZON)] + (['--fail-bank', LOCAL_BANK_PATH, '--fail-neg-alpha', str(ALPHA)] if ALPHA > 0 else [])
if RESUME: cmd.append('--resume')
print(' '.join(cmd)); print('-'*70)
proc = subprocess.Popen(cmd, env={**os.environ}, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
print(f'\n[driver exited {rc}]')

## 11. Mirror checkpoints to Drive

In [ ]:
# 11. Copy checkpoints + metrics to the persistent Drive dir.
import glob, shutil, os, json
n = 0
for f in glob.glob(f'{LOCAL_RUN_DIR}/*.pkl') + glob.glob(f'{LOCAL_RUN_DIR}/*.json') + glob.glob(f'{LOCAL_RUN_DIR}/*.sha256'):
    shutil.copy2(f, f'{RUN_DRIVE_DIR}/{os.path.basename(f)}'); n += 1
print(f'mirrored {n} files to {RUN_DRIVE_DIR}')
mp = f'{LOCAL_RUN_DIR}/metrics.json'
if os.path.exists(mp):
    for e in json.load(open(mp))[-8:]:
        print({k: e.get(k) for k in ('step','success','min_dist','final_dist')})

## 12. Done -- download for the workstation diagnosis

Training + checkpoints are mirrored to Drive. Behavioural characterisation
(route distribution, trigger-avoidance scan, paired mask-flip) runs on the
WORKSTATION: download `best.pkl` (+ `metrics.json`) from the Drive run dir into
`artifacts/naive_rockfall_crl/` and run
`python scripts/diagnose_naive_rockfall.py --ckpt artifacts/naive_rockfall_crl/best.pkl`.


## 12. Colab evaluation = naive-policy diagnosis only, then package
The teacher/center/blind anchors and the authoritative N=1000 pooled table need the scripted walker/base controllers (not committed) and so run on the WORKSTATION after download. On Colab we run the naive diagnosis (route/exposure/drop/leakage/gaming) at cap-800 under the corrected reset, then package the checkpoints for the workstation eval.

In [ ]:
import subprocess, sys, os
os.chdir('/content/'+REPO)
RES = f'{RUN_DRIVE_DIR}/eval_h800_resetfix'
os.makedirs(RES, exist_ok=True)
# 12b. Naive behavioural diagnosis under corrected reset (final + best)
for tag in ('final','best'):
    r = subprocess.run([sys.executable,'scripts/diagnose_naive_rockfall.py',
        '--v2','--p-active','0.30','--reset-fix','--horizon','800','--ckpt', f'{LOCAL_RUN_DIR}/{tag}.pkl',
        '--out-dir', f'{RES}/diag_{tag}'], capture_output=True, text=True)
    print('===',tag,'==='); print(r.stdout[-1200:]); print(r.stderr[-500:] if r.returncode else '')


In [ ]:
# 12c. Package checkpoints + metrics + eval for download
import shutil, glob
pkg = f'/content/{RUN_ID}_bundle'; os.makedirs(pkg, exist_ok=True)
for f in glob.glob(f'{LOCAL_RUN_DIR}/*.pkl')+glob.glob(f'{LOCAL_RUN_DIR}/*.json')+glob.glob(f'{LOCAL_RUN_DIR}/*.sha256'):
    shutil.copy2(f, pkg)
shutil.copytree(RES, f'{pkg}/eval_resetfix', dirs_exist_ok=True)
zp = shutil.make_archive(f'{RUN_DRIVE_DIR}/{RUN_ID}_bundle','zip', pkg)
print('packaged ->', zp)
